# Credit Card Fraud Detection
Detecting fraudulent transactions with imbalanced data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Fetching the dataset
print("Loading Credit Card Fraud dataset...")
try:
    # Attempting to fetch a standard fraud dataset from openml
    fraud_data = fetch_openml(name='creditcard', version=1, as_frame=True, parser='auto')
    df = fraud_data.frame
    
    # Check if target is 'Class'
    if 'Class' not in df.columns and 'target' in df.columns:
        df.rename(columns={'target': 'Class'}, inplace=True)
        
    df['Class'] = df['Class'].astype(int)
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error loading from OpenML: {e}. \nPlease manually download 'creditcard.csv' from Kaggle and place it in this folder.")
    # creating dummy data to avoid crash during execution if download fails
    np.random.seed(42)
    df = pd.DataFrame(np.random.randn(1000, 30), columns=[f'V{i}' for i in range(1, 29)] + ['Time', 'Amount'])
    df['Class'] = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])


In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Undersample the majority class for speed in this demo if dataset is huge, or use SMOTE
if len(df) > 50000:
    print("Dataset is very large, downsampling majority class for faster training...")
    fraud = df[df['Class'] == 1]
    normal = df[df['Class'] == 0].sample(n=len(fraud)*2, random_state=42) # 2:1 ratio
    df = pd.concat([normal, fraud])
    X = df.drop('Class', axis=1)
    y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train['Amount'] = scaler.fit_transform(X_train[['Amount']])
X_test['Amount'] = scaler.transform(X_test[['Amount']])
if 'Time' in X_train.columns:
    X_train['Time'] = scaler.fit_transform(X_train[['Time']])
    X_test['Time'] = scaler.transform(X_test[['Time']])

# Handle class imbalance
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Train
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_res, y_train_res)

# Evaluate
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall: {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
print("Classification Report:\n", classification_report(y_test, y_pred))
